In [ ]:
# Imports and the setup

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.signal import find_peaks
import pywt
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers.legacy import Adam
from tensorflow.keras import backend as K
from sklearn.feature_selection import mutual_info_regression
import ruptures as rpt

np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
# Data loading

def load_cmapss_data(dataset_name='FD001', data_path='CMAPSS/'):
    if not data_path.endswith('/'): 
        data_path += '/'
    index_names = ['unit_id', 'time_cycles']
    setting_names = ['setting_1', 'setting_2', 'setting_3']
    sensor_names = [f'sensor_{i}' for i in range(1, 22)]
    col_names = index_names + setting_names + sensor_names
    train_df = pd.read_csv(f'{data_path}train_{dataset_name}.txt', sep=r'\s+', header=None, names=col_names, engine='python')
    test_df  = pd.read_csv(f'{data_path}test_{dataset_name}.txt',  sep=r'\s+', header=None, names=col_names, engine='python')
    rul_df   = pd.read_csv(f'{data_path}RUL_{dataset_name}.txt',   sep=r'\s+', header=None, names=['RUL'], engine='python')
    print(f"Dataset: {dataset_name} -")
    print(f"Training: {train_df.shape[0]:,} samples | {train_df['unit_id'].nunique()} engines")
    print(f"Test:     {test_df.shape[0]:,} samples | {test_df['unit_id'].nunique()} engines")
    print(f"RUL file: {len(rul_df)} labels")
    return train_df, test_df, rul_df

def add_rul_to_train(train_df):
    max_cycles = train_df.groupby('unit_id')['time_cycles'].max().reset_index()
    max_cycles.columns = ['unit_id', 'max_cycle']
    train_df = train_df.merge(max_cycles, on='unit_id', how='left')
    train_df['RUL'] = train_df['max_cycle'] - train_df['time_cycles']
    train_df = train_df.drop(columns=['max_cycle'])
    return train_df

def prepare_test_data(test_df, rul_df):
    test_last = test_df.groupby('unit_id')['time_cycles'].max().reset_index()
    test_last.columns = ['unit_id', 'last_cycle']
    test_last = test_last.sort_values('unit_id').reset_index(drop=True)
    rul_df = rul_df.copy().reset_index(drop=True)
    if len(rul_df) != len(test_last):
        raise ValueError(f"RUL file has {len(rul_df)} entries, but test has {len(test_last)} engines.")
    test_last['RUL_at_last'] = rul_df['RUL'].values
    test_df = test_df.merge(test_last[['unit_id', 'last_cycle', 'RUL_at_last']], on='unit_id', how='left')
    test_df['RUL'] = test_df['RUL_at_last'] + (test_df['last_cycle'] - test_df['time_cycles'])
    test_df = test_df.drop(columns=['last_cycle', 'RUL_at_last'])
    return test_df

In [ ]:
# Data preprocessing

class Preprocessor:    
    def __init__(self, max_rul=125, use_paper_sensors=True):
        self.max_rul = max_rul
        self.scalers = {}
        self.condition_clusters = None
        self.selected_features = None
        self.feature_importance = {}
        self.degradation_onsets = None
        self.use_paper_sensors = use_paper_sensors
        # Important sensors
        self.paper_sensors = ['sensor_2', 'sensor_3', 'sensor_4', 'sensor_7', 
                              'sensor_8', 'sensor_11', 'sensor_12', 'sensor_13', 
                              'sensor_15', 'sensor_17', 'sensor_20', 'sensor_21']
    def cap_rul(self, df):
        # Linear RUL capping
        df = df.copy()
        df['RUL'] = df['RUL'].clip(upper=self.max_rul)
        return df
    
    def identify_constant_features(self, df):
        # Feature identification
        sensor_cols = [col for col in df.columns if col.startswith('sensor_')]
        constant_features = []
        for col in sensor_cols:
            if df[col].std() < 0.01:
                constant_features.append(col)
        print(f"\nConstant/Low-Variance Features: {constant_features}\n")
        return constant_features
    
    def dual_feature_selection(self, df, target='RUL', pearson_threshold=0.1,
                               cmi_threshold=0.05, min_features=10, max_features=20, variance_threshold=0.95):
        # Option 1 : Important - Pearson only
        if self.use_paper_sensors:
            available_sensors = [s for s in self.paper_sensors if s in df.columns]
            # Removing constant features
            constant_features = self.identify_constant_features(df)
            available_sensors = [s for s in available_sensors if s not in constant_features]
            print(f"Reference Sensors: {len(self.paper_sensors)}")
            print(f"Available in Data: {len([s for s in self.paper_sensors if s in df.columns])}")
            print(f"After Removing Constant: {len(available_sensors)}")
            print(f"Selected Sensors: {available_sensors}")
            # Calculate correlations (Pearson)
            correlations = {}
            for col in available_sensors:
                corr = abs(df[col].corr(df[target]))
                correlations[col] = corr            
            sorted_sensors = sorted(correlations.items(), key=lambda x: x[1], reverse=True)
            print(f"\nSensor Correlations with RUL:")
            for sensor, corr in sorted_sensors:
                print(f"  {sensor}: {corr:.3f}")
            self.feature_importance = {
                'pearson': correlations,
                'mutual_info': {},
                'selected': available_sensors}
            self.selected_features = available_sensors
            return available_sensors
        # Option 2: Adaptive - Dual Feature Selection (Fallback)
        else:
            sensor_cols = [col for col in df.columns if col.startswith('sensor_')]
            setting_cols = [col for col in df.columns if col.startswith('setting_')]
            feature_cols = setting_cols + sensor_cols        
            constant_features = self.identify_constant_features(df)
            feature_cols = [f for f in feature_cols if f not in constant_features]
            # Pearson correlation (Linear)
            correlations = {}
            for col in feature_cols:
                corr = abs(df[col].corr(df[target]))
                correlations[col] = corr
            pearson_selected = [k for k, v in correlations.items() 
                               if v >= pearson_threshold]
            print(f"\nPearson Correlation Analysis:")
            print(f"  Threshold: {pearson_threshold}")
            print(f"  Selected features: {len(pearson_selected)}")
            # Mutual information (Non-linear)
            X = df[feature_cols].values
            y = df[target].values
            mi_scores = mutual_info_regression(X, y, random_state=42)
            mi_dict = dict(zip(feature_cols, mi_scores))
            cmi_selected = [k for k, v in mi_dict.items() if v >= cmi_threshold]
            print(f"\nConditional Mutual Information Analysis:")
            print(f"  Threshold: {cmi_threshold}")
            print(f"  Selected features: {len(cmi_selected)}")
            # Union and adaptive selection
            selected_features = list(set(pearson_selected + cmi_selected))
            if len(selected_features) > max_features:
                # Combine scores (normalized)
                combined_scores = {}
                max_corr = max(correlations.values()) if correlations else 1.0
                max_mi = max(mi_dict.values()) if mi_dict else 1.0
                for feat in selected_features:
                    norm_corr = correlations.get(feat, 0) / max_corr
                    norm_mi = mi_dict.get(feat, 0) / max_mi
                    combined_scores[feat] = (norm_corr + norm_mi) / 2                
                sorted_features = sorted(combined_scores.items(), 
                                       key=lambda x: x[1], reverse=True)
                cumulative_sum = 0
                total_importance = sum(s[1] for s in sorted_features)
                n_features = min_features
                for i, (feat, score) in enumerate(sorted_features):
                    cumulative_sum += score
                    if cumulative_sum / total_importance >= variance_threshold:
                        n_features = min(i + 1, max_features)
                        break
                selected_features = [f[0] for f in sorted_features[:n_features]]
                print(f"\nAdaptive Selection:")
                print(f"  Cumulative Importance Threshold: {variance_threshold*100}%")
                print(f"  Selected: {len(selected_features)} Features")
            elif len(selected_features) < min_features:
                # If few, add top features by combined score
                combined_scores = {}
                max_corr = max(correlations.values()) if correlations else 1.0
                max_mi = max(mi_dict.values()) if mi_dict else 1.0
                for feat in feature_cols:
                    norm_corr = correlations.get(feat, 0) / max_corr
                    norm_mi = mi_dict.get(feat, 0) / max_mi
                    combined_scores[feat] = (norm_corr + norm_mi) / 2
                sorted_features = sorted(combined_scores.items(), key=lambda x: x[1], reverse=True)
                selected_features = [f[0] for f in sorted_features[:min_features]]
                print(f"  Final Count: {len(selected_features)} Features")
            print(f"\nFinal Selected Features: {len(selected_features)}")
            print(f"  Features: {selected_features}")
            self.feature_importance = {
                'pearson': correlations,
                'mutual_info': mi_dict,
                'selected': selected_features}
            self.selected_features = selected_features
            return selected_features
    
    def wavelet_denoise(self, signal, wavelet='db4', level=1):
        # Wavelet denoising     
        coeffs = pywt.wavedec(signal, wavelet, level=level)        
        sigma = np.median(np.abs(coeffs[-1])) / 0.6745
        threshold = sigma * np.sqrt(2 * np.log(len(signal))) * 0.5 
        coeffs_thresh = [coeffs[0]]  # Keep approximation
        for i in range(1, len(coeffs)):
            coeffs_thresh.append(pywt.threshold(coeffs[i], threshold, mode='soft'))
        denoised = pywt.waverec(coeffs_thresh, wavelet)
        if len(denoised) > len(signal):
            denoised = denoised[:len(signal)]
        elif len(denoised) < len(signal):
            denoised = np.pad(denoised, (0, len(signal) - len(denoised)), mode='edge')
        return denoised
    
    def apply_wavelet_denoising(self, df):
        # Applying to all sensor columns
        df_denoised = df.copy()
        sensor_cols = [col for col in df.columns if col.startswith('sensor_')]
        print("\nWavelet Denoising:")
        snr_improvements = []
        for unit_id in df['unit_id'].unique():
            unit_mask = df_denoised['unit_id'] == unit_id
            for col in sensor_cols:
                original = df_denoised.loc[unit_mask, col].values
                if len(original) > 10:
                    denoised = self.wavelet_denoise(original)
                    # Calculate SNR (Signal-to-Noise Ratio)
                    noise = original - denoised
                    signal_power = np.var(denoised)
                    noise_power = np.var(noise)
                    if noise_power > 1e-12 and signal_power > 1e-12:
                        snr = 10 * np.log10(signal_power / noise_power)
                        snr_improvements.append(snr)
                    df_denoised.loc[unit_mask, col] = denoised
        valid_snrs = [s for s in snr_improvements if np.isfinite(s) and s > 0]
        avg_snr = np.mean(valid_snrs) if valid_snrs else 0.0
        print(f"\nAverage SNR: {avg_snr:.2f} dB")
        return df_denoised
    
    def cluster_operating_conditions(self, df, n_clusters=6):
        # Multi-modal operating condition clustering
        setting_cols = ['setting_1', 'setting_2', 'setting_3']
        print("\nCondition-Aware Clustering:")        
        unique_conditions = df[setting_cols].drop_duplicates()
        kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10) 
        # K-Means clustering
        cluster_labels = kmeans.fit_predict(unique_conditions)
        condition_to_cluster = dict(zip(
            unique_conditions.apply(tuple, axis=1),
            cluster_labels))
        df['condition_cluster'] = df[setting_cols].apply(
            lambda x: condition_to_cluster[tuple(x)], axis=1)
        print(f"Number Of Clusters: {n_clusters}")
        print(f"Cluster Distribution:")
        print(df['condition_cluster'].value_counts().sort_index())
        self.condition_clusters = kmeans
        return df
    
    def condition_aware_normalization(self, df, fit=True):
        # Condition-aware normalization
        df = df.copy()
        if self.selected_features is None:
            feature_cols = [col for col in df.columns 
                          if col.startswith('sensor_') or col.startswith('setting_')]
        else:
            feature_cols = self.selected_features
        print("\nCondition-Aware Normalization:")        
        for cluster_id in df['condition_cluster'].unique():
            cluster_mask = df['condition_cluster'] == cluster_id
            cluster_data = df.loc[cluster_mask, feature_cols]
            if fit:
                scaler = StandardScaler()
                scaled_data = scaler.fit_transform(cluster_data)
                self.scalers[cluster_id] = scaler
                print(f"Cluster {cluster_id}: {cluster_mask.sum()} Samples (fitted)")
            else:
                if cluster_id in self.scalers:
                    scaler = self.scalers[cluster_id]
                    scaled_data = scaler.transform(cluster_data)
                    print(f"Cluster {cluster_id}: {cluster_mask.sum()} Samples (transformed)")
                else:
                    print(f"Cluster {cluster_id}: {cluster_mask.sum()} Samples (no scaler, raw)")
                    scaled_data = cluster_data.values
            df.loc[cluster_mask, feature_cols] = scaled_data
        return df
    
    def detect_degradation_onset_pelt(self, signal, penalty=10):
        # Automatic degradation onset detection using PELT algorithm
        algo = rpt.Pelt(model="rbf", min_size=5).fit(signal)
        try:
            changepoints = algo.predict(pen=penalty)
            if len(changepoints) > 1:
                return changepoints[0]
            else:
                return len(signal) // 2
        except:
            return len(signal) // 2
    
    def apply_pelt_degradation_detection(self, df):
        # Apply PELT for each engine
        # NOT used currently for filtering, full engine life used for training
        df = df.copy()        
        print("\nPELT Degradation Onset Detection:")      
        degradation_onsets = {}
        for unit_id in df['unit_id'].unique():
            unit_mask = df['unit_id'] == unit_id
            unit_data = df.loc[unit_mask].sort_values('time_cycles')
            if 'sensor_4' in df.columns:
                signal = unit_data['sensor_4'].values.reshape(-1, 1)
            elif len(self.selected_features) > 0:
                signal = unit_data[self.selected_features[0]].values.reshape(-1, 1)
            else:
                sensor_cols = [col for col in df.columns if col.startswith('sensor_')]
                signal = unit_data[sensor_cols[0]].values.reshape(-1, 1)            
            onset_idx = self.detect_degradation_onset_pelt(signal, penalty=10)
            onset_cycle = unit_data.iloc[onset_idx]['time_cycles']
            degradation_onsets[unit_id] = onset_cycle
        onset_cycles = list(degradation_onsets.values())
        print(f"\nDegradation Onset Statistics:")
        print(f"  Mean: {np.mean(onset_cycles):.1f} cycles")
        print(f"  Std: {np.std(onset_cycles):.1f} cycles")
        print(f"  Min: {np.min(onset_cycles):.1f} cycles")
        print(f"  Max: {np.max(onset_cycles):.1f} cycles")
        # Store for potential use
        self.degradation_onsets = degradation_onsets
        return degradation_onsets
    
    def fit_transform(self, train_df):
        # Preprocessing pipeline for training data
        print("\nStage 1A: Data Preprocessing (Training Data) -")       
        train_df = add_rul_to_train(train_df)
        train_df = self.cap_rul(train_df)        
        self.dual_feature_selection(train_df, target='RUL')        
        train_df = self.apply_wavelet_denoising(train_df)        
        train_df = self.cluster_operating_conditions(train_df)
        train_df = self.condition_aware_normalization(train_df, fit=True)
        # For analysis only, not used for training
        degradation_onsets = self.apply_pelt_degradation_detection(train_df)        
        print("\nTraining Data Preprocessing Completed.")        
        return train_df, degradation_onsets
    
    def transform(self, test_df):
        # Preprocessing pipeline for test data
        print("\nStage 1B: Data Preprocessing (Test Data) -")
        test_df = self.apply_wavelet_denoising(test_df)
        setting_cols = ['setting_1', 'setting_2', 'setting_3']
        test_conditions = test_df[setting_cols].values
        cluster_labels = self.condition_clusters.predict(test_conditions)
        test_df['condition_cluster'] = cluster_labels
        test_df = self.condition_aware_normalization(test_df, fit=False)
        print("\nTest Data Preprocessing Completed.")
        return test_df

In [ ]:
def create_sequences(df, sequence_length, selected_features, degradation_onsets=None):
    # Sliding window sequence creation for deep learning input
    # Uses entire engine life (not just post-degradation)

    # Feature selection
    if selected_features is None:
        feature_cols = [col for col in df.columns 
                       if col.startswith('sensor_') or col.startswith('setting_')]
    else:
        feature_cols = selected_features
    setting_cols = ['setting_1', 'setting_2', 'setting_3']
    # Validation
    missing_settings = [s for s in setting_cols if s not in df.columns]
    if missing_settings:
        raise ValueError(f"Missing setting columns: {missing_settings}")
    missing_features = [f for f in feature_cols if f not in df.columns]
    if missing_features:
        raise ValueError(f"Missing feature columns: {missing_features}")
    X_list, y_list, cond_list, engine_ids = [], [], [], []
    print(f"\nCreating sequences from engine life (len={sequence_length})...")
    skipped_engines = 0
    total_engines = df['unit_id'].nunique()
    for unit_id in df['unit_id'].unique():
        unit_data = df[df['unit_id'] == unit_id].sort_values('time_cycles').copy()
        # Skip if engine life is too short for even one sequence
        if len(unit_data) < sequence_length:
            skipped_engines += 1
            continue
        # Extract arrays
        values = unit_data[feature_cols].values.astype(np.float32)
        rul = unit_data['RUL'].values.astype(np.float32)
        conditions = unit_data[setting_cols].values.astype(np.float32)        
        n_windows = len(values) - sequence_length + 1
        for i in range(n_windows):
            X_list.append(values[i:i + sequence_length])
            y_list.append(rul[i + sequence_length - 1]) 
            cond_list.append(conditions[i + sequence_length - 1])
            engine_ids.append(unit_id)
    # Convert to arrays
    X = np.array(X_list, dtype=np.float32)
    y = np.array(y_list, dtype=np.float32).reshape(-1, 1)
    cond_features = np.array(cond_list, dtype=np.float32)
    engine_ids = np.array(engine_ids)
    print(f"\nSequence Creation Summary:")
    print(f"  Total Engines: {total_engines}")
    print(f"  Engines Used: {total_engines - skipped_engines}")
    print(f"  Engines Skipped (Too Short): {skipped_engines}")
    print(f"  Total Sequences Created: {len(X):,}")
    print(f"\nData Shapes:")
    print(f"  Input (X): {X.shape} (samples, timesteps, features)")
    print(f"  Target (y): {y.shape}")
    print(f"  Conditions: {cond_features.shape}")
    print(f"\nRUL Distribution in Sequences:")
    print(f"  Min RUL: {y.min():.1f} cycles")
    print(f"  Max RUL: {y.max():.1f} cycles")
    print(f"  Mean RUL: {y.mean():.1f} cycles")
    print(f"  Median RUL: {np.median(y):.1f} cycles")
    early = np.sum(y > 80)
    mid = np.sum((y > 30) & (y <= 80))
    critical = np.sum(y <= 30)    
    print(f"\nRUL Range Distribution:")
    print(f"  Early Phase (RUL > 80): {early:,} ({early/len(y)*100:.1f}%)")
    print(f"  Mid Phase (30 < RUL ≤ 80): {mid:,} ({mid/len(y)*100:.1f}%)")
    print(f"  Critical Phase (RUL ≤ 30): {critical:,} ({critical/len(y)*100:.1f}%)")
    return X, y, cond_features, engine_ids

def augment_sequences(X, y, cond, augmentation_factor=3):  
    # Data augmentation with multiple techniques
    X_aug = [X]
    y_aug = [y]
    cond_aug = [cond]
    for i in range(augmentation_factor - 1):
        # Technique 1: Jittering (Gaussian noise)
        noise = np.random.normal(0, 0.005, X.shape)  # Reduced noise
        X_jittered = X + noise
        # Technique 2: Scaling (simulate sensor drift)
        if i == 0:
            scale = np.random.uniform(0.95, 1.05, (1, 1, X.shape[2]))
            X_scaled = X * scale
            X_aug.append(X_scaled)
        else:
            X_aug.append(X_jittered)
        y_aug.append(y)
        cond_aug.append(cond)
    return (np.concatenate(X_aug, axis=0), np.concatenate(y_aug, axis=0),
            np.concatenate(cond_aug, axis=0))

In [ ]:
# Bayesian uncertainty quantification

def negative_log_likelihood(y_true, y_pred, model, weight_decay=1e-4):
    # NLL loss with L2 regularization
    mean = y_pred[:, 0:1]
    log_var = y_pred[:, 1:2]
    log_var = tf.clip_by_value(log_var, -5.0, 3.0)
    variance = tf.exp(log_var)
    variance = tf.maximum(variance, 1e-6)
    precision = 1.0 / (variance + 1e-6)
    squared_error = tf.square(y_true - mean)
    nll = 0.5 * (log_var + squared_error * precision + tf.cast(tf.math.log(2.0 * np.pi), tf.float32))
    # RUL-aware weighting
    rul_weights = tf.where(
        y_true < 30.0,
        2.5,  # Very high weight for critical
        tf.where(
            y_true < 80.0,
            1.5,  # High weight for near-critical
            1.0))   # Normal weight for early
    weighted_nll = nll * rul_weights
    # Uncertainty penalty
    uncertainty_penalty = 0.005 * tf.reduce_mean(tf.square(log_var))
    # Manual L2 regularization
    l2_loss = tf.add_n([tf.nn.l2_loss(v) for v in model.trainable_variables 
                        if 'kernel' in v.name])
    total_loss = tf.reduce_mean(weighted_nll) + uncertainty_penalty + weight_decay * l2_loss
    return total_loss

class BayesianOutputLayer(layers.Layer):
    # Bayesian output layer with variance constraints    
    def __init__(self, **kwargs):
        super(BayesianOutputLayer, self).__init__(**kwargs)
        self.mean_dense = layers.Dense(
            1, kernel_initializer='glorot_uniform',
            name='mean_predictor')
        self.log_var_dense = layers.Dense(
            1, kernel_initializer='zeros',
            bias_initializer=tf.keras.initializers.Constant(1.0),
            name='uncertainty_predictor')
    def call(self, inputs):
        mean = self.mean_dense(inputs)
        log_var = self.log_var_dense(inputs)
        log_var = tf.clip_by_value(log_var, -5.0, 3.0)
        return tf.concat([mean, log_var], axis=-1)
    def get_config(self):
        config = super().get_config()
        return config

class MCDropout(layers.Layer):
    # Monte Carlo Dropout for uncertainty estimation (not used)
    def __init__(self, rate, **kwargs):
        super(MCDropout, self).__init__(**kwargs)
        self.rate = rate
        self.dropout = layers.Dropout(rate)
    def call(self, inputs, training=None):
        return self.dropout(inputs, training=True)
    def get_config(self):
        config = super().get_config()
        config.update({"rate": self.rate})
        return config

In [ ]:
# Hierarchical feature extraction

class OperatingConditionEncoder(layers.Layer): 
    # Encodes operating conditions (altitude, Mach, throttle) into learned representation
    def __init__(self, embedding_dim=32, **kwargs): 
        super(OperatingConditionEncoder, self).__init__(**kwargs)
        self.embedding_dim = embedding_dim        
        self.dense1 = layers.Dense(64, activation='relu')
        self.bn1 = layers.BatchNormalization()
        self.dropout1 = layers.Dropout(0.2)
        self.dense2 = layers.Dense(embedding_dim, activation='relu')
        self.bn2 = layers.BatchNormalization()
    def call(self, inputs, training=False):
        x = self.dense1(inputs)
        x = self.bn1(x, training=training)
        x = self.dropout1(x, training=training)
        x = self.dense2(x)
        x = self.bn2(x, training=training)
        return x
    def get_config(self):
        config = super().get_config()
        config.update({"embedding_dim": self.embedding_dim})
        return config

class MultiScaleInceptionBlock(layers.Layer):
    # Multi-scale 1D CNN with parallel branches
    # Captures short, medium, and long-term patterns simultaneously
    def __init__(self, filters=32, **kwargs):
        super(MultiScaleInceptionBlock, self).__init__(**kwargs)
        self.filters = filters
        # Branch 1: Small kernel for local correlations (3-cycle patterns)
        self.conv1 = layers.Conv1D(filters, kernel_size=3, padding='same', 
                                   activation='relu')
        self.bn1 = layers.BatchNormalization()
        # Branch 2: Medium kernel for medium-term trends (5-cycle patterns)
        self.conv2 = layers.Conv1D(filters, kernel_size=5, padding='same', 
                                   activation='relu')
        self.bn2 = layers.BatchNormalization()
        # Branch 3: Large kernel for long-term degradation (7-cycle patterns)
        self.conv3 = layers.Conv1D(filters, kernel_size=7, padding='same', 
                                   activation='relu')
        self.bn3 = layers.BatchNormalization()
        # Concatenate all branches
        self.concat = layers.Concatenate()
        # Max pooling for downsampling
        self.pool = layers.MaxPooling1D(pool_size=2, padding='same')
    def call(self, inputs):
        # Apply three parallel convolutions
        x1 = self.conv1(inputs)
        x1 = self.bn1(x1)
        x2 = self.conv2(inputs)
        x2 = self.bn2(x2)
        x3 = self.conv3(inputs)
        x3 = self.bn3(x3)
        # Concatenate
        x = self.concat([x1, x2, x3])
        # Pooling
        x = self.pool(x)
        return x
    def get_config(self):
        config = super().get_config()
        config.update({"filters": self.filters})
        return config

class MultiHeadDualAttention(layers.Layer):
    # Captures sensor-level and temporal-level patterns simultaneously
    def __init__(self, d_model=128, num_heads=4, **kwargs):
        super().__init__(**kwargs)
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        # Sensor attention
        self.W_q_sensor = [layers.Dense(self.d_k, use_bias=False) for _ in range(num_heads)]
        self.W_k_sensor = [layers.Dense(self.d_k, use_bias=False) for _ in range(num_heads)]
        self.W_v_sensor = [layers.Dense(self.d_k, use_bias=False) for _ in range(num_heads)]
        # Temporal attention
        self.W_q_temporal = [layers.Dense(self.d_k, use_bias=False) for _ in range(num_heads)]
        self.W_k_temporal = [layers.Dense(self.d_k, use_bias=False) for _ in range(num_heads)]
        self.W_v_temporal = [layers.Dense(self.d_k, use_bias=False) for _ in range(num_heads)]
        # Output projection
        self.W_o = layers.Dense(d_model)
        # Learnable fusion
        self.W_fusion = self.add_weight(
            name='fusion_weights', shape=(2,),
            initializer='glorot_uniform', trainable=True)
    def call(self, inputs):
        batch_size = tf.shape(inputs)[0]
        # Sensor attention
        sensor_heads = []
        for i in range(self.num_heads):
            Q_s = self.W_q_sensor[i](inputs)
            K_s = self.W_k_sensor[i](inputs)
            V_s = self.W_v_sensor[i](inputs)
            scores_s = tf.matmul(Q_s, K_s, transpose_b=True) / tf.sqrt(tf.cast(self.d_k, tf.float32))
            attention_s = tf.nn.softmax(scores_s, axis=-1)
            head_s = tf.matmul(attention_s, V_s)
            sensor_heads.append(head_s)
        sensor_context = tf.concat(sensor_heads, axis=-1)
        # Temporal attention
        temporal_heads = []
        for i in range(self.num_heads):
            Q_t = self.W_q_temporal[i](inputs)
            K_t = self.W_k_temporal[i](inputs)
            V_t = self.W_v_temporal[i](inputs)
            scores_t = tf.matmul(Q_t, K_t, transpose_b=True) / tf.sqrt(tf.cast(self.d_k, tf.float32))
            attention_t = tf.nn.softmax(scores_t, axis=-1)
            head_t = tf.matmul(attention_t, V_t)
            temporal_heads.append(head_t)
        temporal_context = tf.concat(temporal_heads, axis=-1)
        # Adaptive fusion
        fusion_weights = tf.nn.softmax(self.W_fusion, axis=0)
        w_sensor = fusion_weights[0]
        w_temporal = fusion_weights[1]
        combined = w_sensor * sensor_context + w_temporal * temporal_context
        # Output projection
        output = self.W_o(combined)
        return output
    def get_config(self):
        config = super().get_config()
        config.update({
            "d_model": self.d_model, "num_heads": self.num_heads})
        return config

In [ ]:
def build_comprehensive_rul_model(sequence_length, n_features, 
                                  use_mc_dropout=False,
                                  dropout_rate=0.48):
    # Deep Learning RUL estimation model
    print("Building RUL Model:")    
    sequence_input = Input(shape=(sequence_length, n_features), name='sequence_input')
    condition_input = Input(shape=(3,), name='condition_input')
    print("\nStage 1: Multi-Scale CNN (Single Block)")
    x = MultiScaleInceptionBlock(filters=32)(sequence_input)
    x = layers.BatchNormalization()(x)
    if use_mc_dropout:
        x = MCDropout(dropout_rate)(x)
    else:
        x = layers.Dropout(dropout_rate)(x)
    print("Stage 2: Bi-Directional LSTM (Single Layer)")
    x = layers.Bidirectional(
        layers.LSTM(64, return_sequences=True, activation='tanh',
                   recurrent_dropout=0.2))(x)
    x = layers.BatchNormalization()(x)
    if use_mc_dropout:
        x = MCDropout(dropout_rate)(x)
    else:
        x = layers.Dropout(dropout_rate)(x)
    print("Stage 3: Dual-Level Attention")
    x = MultiHeadDualAttention(d_model=128, num_heads=1)(x)
    x = layers.GlobalAveragePooling1D()(x)
    print("Stage 4: Operating Condition Encoder")
    condition_encoded = OperatingConditionEncoder(embedding_dim=32)(condition_input)
    x = layers.Concatenate()([x, condition_encoded])
    print("Stage 5: Dense Layers")
    x = layers.Dense(64, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    if use_mc_dropout:
        x = MCDropout(dropout_rate)(x)
    else:
        x = layers.Dropout(dropout_rate)(x)
    x = layers.Dense(32, activation='relu')(x)
    if use_mc_dropout:
        x = MCDropout(dropout_rate)(x)
    else:
        x = layers.Dropout(dropout_rate)(x)
    print("Stage 6: Bayesian Output Layer")
    bayesian_output = BayesianOutputLayer(name='bayesian_output')(x)
    model = Model(
        inputs=[sequence_input, condition_input],
        outputs=bayesian_output, name='RUL_Model')
    print("\nModel Building Completed.")
    print(f"Total parameters: {model.count_params():,}")   
    return model

In [ ]:
# Uncertainty-aware model trainer

class UncertaintyAwareTrainer:    
    def __init__(self, model, learning_rate=0.00015):
        self.model = model
        self.optimizer = Adam(learning_rate=learning_rate, clipnorm=1.0)
        self.train_losses = []
        self.val_losses = []
        self.train_rmses = []
        self.val_rmses = []

    @tf.function
    def train_step(self, X_seq, X_cond, y_true):
        with tf.GradientTape() as tape:
            y_pred = self.model([X_seq, X_cond], training=True)
            loss = negative_log_likelihood(y_true, y_pred, self.model, weight_decay=1e-4)
        gradients = tape.gradient(loss, self.model.trainable_variables)
        self.optimizer.apply_gradients(zip(gradients, self.model.trainable_variables))
        mean_pred = y_pred[:, 0:1]
        rmse = tf.sqrt(tf.reduce_mean(tf.square(y_true - mean_pred)))
        return loss, rmse

    @tf.function
    def val_step(self, X_seq, X_cond, y_true):
        y_pred = self.model([X_seq, X_cond], training=False)
        loss = negative_log_likelihood(y_true, y_pred, self.model, weight_decay=1e-4)
        mean_pred = y_pred[:, 0:1]
        rmse = tf.sqrt(tf.reduce_mean(tf.square(y_true - mean_pred))) 
        return loss, rmse
        
    def train(self, X_train, cond_train, y_train,
              X_val, cond_val, y_val,
              epochs=250, batch_size=64, verbose=1):
        n_batches = len(X_train) // batch_size
        print(f"Training Samples: {len(X_train)}")
        print(f"Validation Samples: {len(X_val)}")
        print(f"Epochs: {epochs}")
        print(f"Batch Size: {batch_size}")
        print(f"Batches per Epoch: {n_batches}")        
        best_val_rmse = float('inf')
        best_val_loss = float('inf')
        patience = 20  
        patience_counter = 0
        lr_patience = 10  
        lr_patience_counter = 0
        weights_path = 'best_model.weights.h5'
        for epoch in range(epochs):
            # Shuffle training data
            indices = np.random.permutation(len(X_train))
            X_train_shuffled = X_train[indices]
            cond_train_shuffled = cond_train[indices]
            y_train_shuffled = y_train[indices]
            # Training
            epoch_losses = []
            epoch_rmses = []
            for i in range(n_batches):
                start_idx = i * batch_size
                end_idx = start_idx + batch_size
                X_batch = X_train_shuffled[start_idx:end_idx]
                cond_batch = cond_train_shuffled[start_idx:end_idx]
                y_batch = y_train_shuffled[start_idx:end_idx]
                loss, rmse = self.train_step(X_batch, cond_batch, y_batch)
                epoch_losses.append(loss.numpy())
                epoch_rmses.append(rmse.numpy())
            avg_train_loss = np.mean(epoch_losses)
            avg_train_rmse = np.mean(epoch_rmses)
            self.train_losses.append(avg_train_loss)
            self.train_rmses.append(avg_train_rmse)
            # Validation
            val_losses = []
            val_rmses = []
            val_batches = len(X_val) // batch_size
            for i in range(val_batches):
                start_idx = i * batch_size
                end_idx = start_idx + batch_size
                X_batch = X_val[start_idx:end_idx]
                cond_batch = cond_val[start_idx:end_idx]
                y_batch = y_val[start_idx:end_idx]
                loss, rmse = self.val_step(X_batch, cond_batch, y_batch)
                val_losses.append(loss.numpy())
                val_rmses.append(rmse.numpy())
            avg_val_loss = np.mean(val_losses)
            avg_val_rmse = np.mean(val_rmses)
            self.val_losses.append(avg_val_loss)
            self.val_rmses.append(avg_val_rmse)
            # Print progress
            if verbose and (epoch + 1) % 5 == 0:
                current_lr = float(tf.keras.backend.get_value(self.optimizer.learning_rate))
                print(f"Epoch {epoch+1}/{epochs}")
                print(f"  Train - Loss: {avg_train_loss:.4f}, RMSE: {avg_train_rmse:.2f}")
                print(f"  Val   - Loss: {avg_val_loss:.4f}, RMSE: {avg_val_rmse:.2f}")
                print(f"  LR: {current_lr:.2e}")
            # Save best model
            if avg_val_rmse < best_val_rmse:
                best_val_rmse = avg_val_rmse
                best_val_loss = avg_val_loss
                patience_counter = 0
                lr_patience_counter = 0
                self.model.save_weights(weights_path)
                if verbose:
                    print(f"  Saved Best Model (RMSE: {best_val_rmse:.2f}, NLL: {best_val_loss:.4f})")
            else:
                patience_counter += 1
                lr_patience_counter += 1
                # Reduce LR less aggressively
                if lr_patience_counter >= lr_patience:
                    old_lr = float(self.optimizer.learning_rate.numpy())
                    new_lr = max(old_lr * 0.5, 1e-6)
                    self.optimizer.learning_rate.assign(new_lr)
                    print(f"\n  [ReduceLROnPlateau] LR Reduced: {old_lr:.2e} → {new_lr:.2e}")
                    lr_patience_counter = 0
            # Early Stopping
            if patience_counter >= patience:
                print(f"\nEarly Stopping at Epoch {epoch+1}")
                print(f"Best Validation RMSE: {best_val_rmse:.2f}")
                print(f"Best Validation NLL: {best_val_loss:.4f}")
                break
        # Load best weights
        if os.path.exists(weights_path):
            self.model.load_weights(weights_path)
            print("\nLoading Best Model Weights...")
            print(f"Final Validation RMSE: {best_val_rmse:.2f}")
        return {
            'train_losses': self.train_losses, 'val_losses': self.val_losses,
            'train_rmses': self.train_rmses, 'val_rmses': self.val_rmses,
            'best_val_rmse': best_val_rmse, 'best_val_loss': best_val_loss}

In [ ]:
def predict_deterministic(model, X_seq, X_cond):
    # Deterministic prediction with learned uncertainty    
    # Single forward pass with dropout OFF (training=False)
    y_pred = model([X_seq, X_cond], training=False)
    # Extract predictions and learned uncertainty
    mean = y_pred[:, 0].numpy()
    log_var = y_pred[:, 1].numpy()    
    log_var = np.clip(log_var, -5.0, 3.0)
    variance = np.exp(log_var)
    std = np.sqrt(variance)
    lower_95 = mean - 1.96 * std
    upper_95 = mean + 1.96 * std
    return {'mean': mean, 'std': std, 'aleatoric': std, 
        'epistemic': np.zeros_like(std), 'lower_95': lower_95, 'upper_95': upper_95}

In [ ]:
# Evaluation metrics

class ComprehensiveEvaluator:
    def __init__(self):
        self.metrics = {}
    def calculate_rmse(self, y_true, y_pred):
        # Root Mean Squared Error
        return np.sqrt(mean_squared_error(y_true, y_pred))
    def calculate_mae(self, y_true, y_pred):
        # Mean Absolute Error
        return mean_absolute_error(y_true, y_pred)
    def calculate_mape(self, y_true, y_pred):
        # Mean Absolute Percentage Error
        epsilon = 1e-10
        return np.mean(np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), epsilon))) * 100
    def calculate_r2(self, y_true, y_pred):
        # R-Squared Score
        return r2_score(y_true, y_pred)
        errors = y_pred - y_true
        scores = []
        for error in errors:
            if error < 0:  # Early prediction 
                scores.append(np.exp(-error / 13) - 1)
            else:  # Late prediction
                scores.append(np.exp(error / 10) - 1)
        return np.sum(scores)
    
    def calculate_coverage_at_confidence_levels(self, y_true, y_pred_mean, y_pred_std):
        # Calculate coverage at multiple confidence levels
        confidence_levels = [0.90, 0.95, 0.99]
        z_scores = [1.645, 1.96, 2.576]  # Corresponding z-scores
        coverages = {}
        calibration_errors = []
        for conf_level, z in zip(confidence_levels, z_scores):
            lower = y_pred_mean - z * y_pred_std
            upper = y_pred_mean + z * y_pred_std
            within_bounds = np.logical_and(y_true >= lower, y_true <= upper)
            coverage = np.mean(within_bounds)
            coverages[f'{int(conf_level*100)}%'] = coverage * 100
            calibration_errors.append(abs(coverage - conf_level))
        mean_calibration_error = np.mean(calibration_errors)
        return coverages, mean_calibration_error  
    
    def evaluate_by_rul_range(self, y_true, y_pred_mean, y_pred_std=None):
        # Evaluate model performance by RUL range        
        ranges = {
            'Early (RUL > 80)': (80, float('inf')),
            'Mid (30 < RUL ≤ 80)': (30, 80),
            'Critical (RUL ≤ 30)': (0, 30)}
        range_metrics = {}
        print("\n3. Performance By RUL Range:")        
        for range_name, (low, high) in ranges.items():
            mask = (y_true > low) & (y_true <= high)
            n_samples = np.sum(mask)
            if n_samples > 0:
                rmse = self.calculate_rmse(y_true[mask], y_pred_mean[mask])
                mae = self.calculate_mae(y_true[mask], y_pred_mean[mask])
                range_metrics[range_name] = {
                    'n_samples': n_samples, 'rmse': rmse, 'mae': mae}
                print(f"\n{range_name}:")
                print(f"  Samples: {n_samples}")
                print(f"  RMSE: {rmse:.2f}")
                print(f"  MAE: {mae:.2f}")
                if y_pred_std is not None:
                    avg_uncertainty = np.mean(y_pred_std[mask])
                    range_metrics[range_name]['avg_uncertainty'] = avg_uncertainty
        return range_metrics
    
    def evaluate_all(self, y_true, y_pred, y_pred_std=None, 
                     lower_bound=None, upper_bound=None):
        # Evaluate all metrics       
        print("\nComprehensive Evaluation Results:")        
        metrics = {}
        print("\n1. Accuracy Metrics:")        
        metrics['RMSE'] = np.sqrt(mean_squared_error(y_true, y_pred))
        metrics['MAE'] = np.mean(np.abs(y_true - y_pred))
        metrics['MAPE'] = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-8))) * 100
        metrics['R2'] = r2_score(y_true, y_pred)
        print(f"  RMSE: {metrics['RMSE']:.4f}")
        print(f"  MAE:  {metrics['MAE']:.4f}")
        print(f"  MAPE: {metrics['MAPE']:.2f}%")
        print(f"  R²:   {metrics['R2']:.4f}")
        if y_pred_std is not None:
            print("\n2. Uncertainty Metrics:")
            coverages, mean_cal_error = self.calculate_coverage_at_confidence_levels(
                y_true, y_pred, y_pred_std)
            print("\nConfidence Interval Coverage:")
            for level, coverage in coverages.items():
                expected = float(level.strip('%'))
                diff = coverage - expected
                print(f"  {level} CI: {coverage:.1f}% (expected: {expected}%, diff: {diff:+.1f}%)")
            metrics['Coverages'] = coverages          
            # Performance by RUL range
            range_metrics = self.evaluate_by_rul_range(y_true, y_pred, y_pred_std)
            metrics['Range_Metrics'] = range_metrics
        return metrics
    
    def plot_results(self, y_true, y_pred_mean, y_pred_std=None, 
                    lower_bound=None, upper_bound=None, save_path=None):
        # Comprehensive visualizations
        y_true = y_true.flatten()
        y_pred_mean = y_pred_mean.flatten()
        if y_pred_std is not None:
            y_pred_std = y_pred_std.flatten()
        if lower_bound is not None:
            lower_bound = lower_bound.flatten()
        if upper_bound is not None:
            upper_bound = upper_bound.flatten()
        fig = plt.figure(figsize=(18, 12))
        gs = fig.add_gridspec(1, 3, hspace=0.3, wspace=0.3)

        # 1. Predicted vs Actual
        ax1 = fig.add_subplot(gs[0, 0])
        ax1.scatter(y_true, y_pred_mean, alpha=0.5, s=10)
        ax1.plot([y_true.min(), y_true.max()], 
                [y_true.min(), y_true.max()], 
                'r--', lw=2, label='Perfect Prediction')
        ax1.set_xlabel('True RUL', fontsize=12)
        ax1.set_ylabel('Predicted RUL', fontsize=12)
        ax1.set_title('Predicted vs Actual RUL', fontsize=14, fontweight='bold')
        ax1.legend()
        ax1.grid(True, alpha=0.3)       
      
        # 2. Residual Plot
        ax2 = fig.add_subplot(gs[0, 1])
        errors = y_pred_mean - y_true
        ax2.scatter(y_pred_mean, errors, alpha=0.5, s=10, c=y_true, cmap='viridis')
        ax2.axhline(0, color='r', linestyle='--', linewidth=2)
        ax2.set_xlabel('Predicted RUL', fontsize=12)
        ax2.set_ylabel('Residuals (Predicted - True)', fontsize=12)
        ax2.set_title('Residual Plot', fontsize=14, fontweight='bold')
        ax2.grid(True, alpha=0.3)
        cbar = plt.colorbar(ax2.collections[0], ax=ax2)
        cbar.set_label('True RUL', rotation=270, labelpad=15)
    
        # 3. Uncertainty Visualization
        if y_pred_std is not None and lower_bound is not None:
            ax3 = fig.add_subplot(gs[0, 2])    
            sorted_indices = np.argsort(y_true)
            y_true_sorted = y_true[sorted_indices]
            y_pred_sorted = y_pred_mean[sorted_indices]
            lower_sorted = lower_bound[sorted_indices]
            upper_sorted = upper_bound[sorted_indices]            
            step = max(1, len(y_true_sorted) // 100)
            indices = np.arange(0, len(y_true_sorted), step)
            ax3.plot(indices, y_true_sorted[indices], 'b-', 
                    label='True RUL', linewidth=2, alpha=0.7)
            ax3.plot(indices, y_pred_sorted[indices], 'r-', 
                    label='Predicted RUL', linewidth=2, alpha=0.7)
            ax3.fill_between(indices, lower_sorted[indices], upper_sorted[indices],
                            alpha=0.3, color='red', label='95% Confidence Interval')
            ax3.set_xlabel('Sample Index (sorted by true RUL)', fontsize=12)
            ax3.set_ylabel('RUL (cycles)', fontsize=12)
            ax3.set_title('Predictions with Uncertainty Bounds', 
                         fontsize=14, fontweight='bold')
            ax3.legend()
            ax3.grid(True, alpha=0.3)            
        else:
            # If no uncertainty available
            ax4 = fig.add_subplot(gs[1, 0])
            ax4.text(0.5, 0.5, 'Uncertainty metrics not available',
                    ha='center', va='center', fontsize=12)
            ax4.axis('off')
            ax5 = fig.add_subplot(gs[1, 1])
            ax5.text(0.5, 0.5, 'Uncertainty metrics not available',
                    ha='center', va='center', fontsize=12)
            ax5.axis('off')
            ax6 = fig.add_subplot(gs[1, 2])
            ax6.text(0.5, 0.5, 'Uncertainty metrics not available',
                    ha='center', va='center', fontsize=12)
            ax6.axis('off')
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            print(f"\nPlot saved: {save_path}")
        plt.show()

In [ ]:
# False positive / false negative analysis for predictive maintenance

class MaintenanceDecisionAnalyzer:
    def __init__(self, critical_threshold=30, warning_threshold=50):
        self.critical_threshold = critical_threshold
        self.warning_threshold = warning_threshold
    def classify_maintenance_decision(self, rul_values, threshold):
        return (rul_values <= threshold).astype(int)

    def calculate_fp_fn_metrics(self, y_true, y_pred, threshold):
        # Calculate false positives, false negatives, and related metrics
        true_labels = self.classify_maintenance_decision(y_true, threshold)
        pred_labels = self.classify_maintenance_decision(y_pred, threshold)
        # Calculate confusion matrix components
        tp = np.sum((true_labels == 1) & (pred_labels == 1))  
        tn = np.sum((true_labels == 0) & (pred_labels == 0))  
        fp = np.sum((true_labels == 0) & (pred_labels == 1))  
        fn = np.sum((true_labels == 1) & (pred_labels == 0)) 
        total = len(y_true)
        # Calculate rates
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0  
        fnr = fn / (fn + tp) if (fn + tp) > 0 else 0  
        tpr = tp / (tp + fn) if (tp + fn) > 0 else 0  
        tnr = tn / (tn + fp) if (tn + fp) > 0 else 0 
        # Additional metrics
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        f1_score = 2 * (precision * tpr) / (precision + tpr) if (precision + tpr) > 0 else 0
        accuracy = (tp + tn) / total if total > 0 else 0
        return {
            'threshold': threshold, 'TP': tp, 'TN': tn, 'FP': fp, 'FN': fn,
            'FPR': fpr, 'FNR': fnr, 'TPR': tpr, 'TNR': tnr,
            'Precision': precision, 'Recall': tpr, 'F1_Score': f1_score,
            'Accuracy': accuracy, 'Total': total}
    
    def analyze_multiple_thresholds(self, y_true, y_pred, thresholds=None):
        if thresholds is None:
            thresholds = [20, 30, 40, 50, 60]
        results = []
        for threshold in thresholds:
            metrics = self.calculate_fp_fn_metrics(y_true, y_pred, threshold)
            results.append(metrics)
        return pd.DataFrame(results)
    
    def analyze_prediction_errors(self, y_true, y_pred):
        errors = y_pred - y_true        
        over_predictions = errors > 0  
        under_predictions = errors < 0 
        over_pred_errors = errors[over_predictions]
        under_pred_errors = errors[under_predictions]        
        results = {
            'Over-predictions (Dangerous)': {
                'count': np.sum(over_predictions),
                'percentage': np.mean(over_predictions) * 100,
                'mean_error': np.mean(over_pred_errors) if len(over_pred_errors) > 0 else 0,
                'max_error': np.max(over_pred_errors) if len(over_pred_errors) > 0 else 0,},
            'Under-predictions (Unnecessary Maintenance)': {
                'count': np.sum(under_predictions),
                'percentage': np.mean(under_predictions) * 100,
                'mean_error': np.abs(np.mean(under_pred_errors)) if len(under_pred_errors) > 0 else 0,
                'max_error': np.abs(np.min(under_pred_errors)) if len(under_pred_errors) > 0 else 0,}}
        return results
    
    def plot_fp_fn_analysis(self, y_true, y_pred, save_path=None):
        # Visualize false positive and false negative analysis
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))

        # 1. Confusion Matrix for Critical Threshold
        ax1 = axes[0, 0]
        metrics = self.calculate_fp_fn_metrics(y_true, y_pred, self.critical_threshold)
        confusion = np.array([[metrics['TN'], metrics['FP']], 
                             [metrics['FN'], metrics['TP']]])
        sns.heatmap(confusion, annot=True, fmt='d', cmap='Blues', ax=ax1,
                   xticklabels=['Healthy (Pred)', 'Maintenance (Pred)'],
                   yticklabels=['Healthy (True)', 'Maintenance (True)'],
                   cbar_kws={'label': 'Count'})
        ax1.set_title(f'Confusion Matrix (Threshold: RUL ≤ {self.critical_threshold})', 
                     fontsize=14, fontweight='bold')
        ax1.set_ylabel('True Condition', fontsize=12)
        ax1.set_xlabel('Predicted Condition', fontsize=12)        
        ax1.text(0.5, -0.15, f'FP (False Alarms): {metrics["FP"]} | FN (Missed): {metrics["FN"]}',
                transform=ax1.transAxes, ha='center', fontsize=11, 
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        
        # 2. FPR and FNR across thresholds
        ax2 = axes[0, 1]
        thresholds = np.arange(10, 101, 5)
        results = self.analyze_multiple_thresholds(y_true, y_pred, thresholds)
        
        ax2.plot(results['threshold'], results['FPR'] * 100, 
                marker='o', label='False Positive Rate', linewidth=2, color='orange')
        ax2.plot(results['threshold'], results['FNR'] * 100, 
                marker='s', label='False Negative Rate', linewidth=2, color='red')
        ax2.axvline(x=self.critical_threshold, color='green', linestyle='--', 
                   label=f'Critical Threshold ({self.critical_threshold})')
        ax2.set_xlabel('RUL Threshold (cycles)', fontsize=12)
        ax2.set_ylabel('Error Rate (%)', fontsize=12)
        ax2.set_title('FPR and FNR vs RUL Threshold', fontsize=14, fontweight='bold')
        ax2.legend()
        ax2.grid(True, alpha=0.3)
        
        # 3. Precision-Recall Trade-off
        ax3 = axes[1, 0]
        ax3.plot(results['threshold'], results['Precision'] * 100, 
                marker='o', label='Precision', linewidth=2, color='blue')
        ax3.plot(results['threshold'], results['Recall'] * 100, 
                marker='s', label='Recall (TPR)', linewidth=2, color='green')
        ax3.plot(results['threshold'], results['F1_Score'] * 100, 
                marker='^', label='F1 Score', linewidth=2, color='purple')
        ax3.axvline(x=self.critical_threshold, color='red', linestyle='--', 
                   label=f'Critical Threshold ({self.critical_threshold})')
        ax3.set_xlabel('RUL Threshold (cycles)', fontsize=12)
        ax3.set_ylabel('Score (%)', fontsize=12)
        ax3.set_title('Precision-Recall Trade-off', fontsize=14, fontweight='bold')
        ax3.legend()
        ax3.grid(True, alpha=0.3)
        
        # 4. Error Distribution
        ax4 = axes[1, 1]
        errors = y_pred - y_true        
        over_mask = errors > 0
        under_mask = errors < 0
        ax4.hist(errors[over_mask], bins=30, alpha=0.6, color='red', 
                label=f'Over-predictions (n={np.sum(over_mask)})\nDangerous!', edgecolor='black')
        ax4.hist(errors[under_mask], bins=30, alpha=0.6, color='blue', 
                label=f'Under-predictions (n={np.sum(under_mask)})\nUnnecessary Maintenance', 
                edgecolor='black')
        ax4.axvline(x=0, color='green', linestyle='--', linewidth=2, label='Perfect Prediction')
        ax4.set_xlabel('Prediction Error (Predicted - True RUL)', fontsize=12)
        ax4.set_ylabel('Frequency', fontsize=12)
        ax4.set_title('Error Distribution Analysis', fontsize=14, fontweight='bold')
        ax4.legend(loc='upper left', fontsize=10)
        ax4.grid(True, alpha=0.3)
        plt.tight_layout()
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            print(f"\nPlot saved: {save_path}")
        plt.show()
    
    def print_comprehensive_report(self, y_true, y_pred):
        print(f"\n1. Maintenance Decision Analysis (Critical Threshold: RUL ≤ {self.critical_threshold}) -")
        metrics = self.calculate_fp_fn_metrics(y_true, y_pred, self.critical_threshold)
        print(f"\nConfusion Matrix:")
        print(f"  True Positives (TP):  {metrics['TP']:4d}")
        print(f"  True Negatives (TN):  {metrics['TN']:4d}")
        print(f"  False Positives (FP): {metrics['FP']:4d}")
        print(f"  False Negatives (FN): {metrics['FN']:4d}")
        print(f"\nPerformance Metrics:")
        print(f"  Accuracy:             {metrics['Accuracy']*100:6.2f}%")
        print(f"  Precision:            {metrics['Precision']*100:6.2f}%")
        print(f"  Recall (Sensitivity): {metrics['Recall']*100:6.2f}%")
        print(f"  F1 Score:             {metrics['F1_Score']*100:6.2f}%")
        print(f"\nError Rates:")
        print(f"  False Positive Rate:  {metrics['FPR']*100:6.2f}%")
        print(f"  False Negative Rate:  {metrics['FNR']*100:6.2f}%")
        print(f"  True Positive Rate:   {metrics['TPR']*100:6.2f}%")
        print(f"  True Negative Rate:   {metrics['TNR']*100:6.2f}%")
        
        print(f"\n2. Prediction Error Analysis -")
        error_analysis = self.analyze_prediction_errors(y_true, y_pred)
        print(f"\nOver-predictions:")
        print(f"  Count:          {error_analysis['Over-predictions (Dangerous)']['count']}")
        print(f"  Percentage:     {error_analysis['Over-predictions (Dangerous)']['percentage']:.2f}%")
        print(f"  Mean Error:     {error_analysis['Over-predictions (Dangerous)']['mean_error']:.2f} cycles")
        print(f"  Max Error:      {error_analysis['Over-predictions (Dangerous)']['max_error']:.2f} cycles")
        print(f"\nUnder-predictions:")
        print(f"  Count:          {error_analysis['Under-predictions (Unnecessary Maintenance)']['count']}")
        print(f"  Percentage:     {error_analysis['Under-predictions (Unnecessary Maintenance)']['percentage']:.2f}%")
        print(f"  Mean Error:     {error_analysis['Under-predictions (Unnecessary Maintenance)']['mean_error']:.2f} cycles")
        print(f"  Max Error:      {error_analysis['Under-predictions (Unnecessary Maintenance)']['max_error']:.2f} cycles")
        
        print(f"\n3. Multi-Threshold Analysis -")
        thresholds = [20, 30, 40, 50]
        results_df = self.analyze_multiple_thresholds(y_true, y_pred, thresholds)
        print(f"\n{'Threshold':>10} | {'FP':>6} | {'FN':>6} | {'FPR':>7} | {'FNR':>7} | {'F1':>7}")
        print("-" * 60)
        for _, row in results_df.iterrows():
            print(f"{row['threshold']:>10.0f} | {row['FP']:>6.0f} | {row['FN']:>6.0f} | "
                  f"{row['FPR']*100:>6.2f}% | {row['FNR']*100:>6.2f}% | {row['F1_Score']*100:>6.2f}%")
        print("\n")
        return metrics

In [ ]:
def main_pipeline(dataset_name='FD001', data_path='', 
                 sequence_length=30, epochs=250):
    # Complete end-to-end pipeline
    print("\n\nStarting Pipeline Execution...")

    print("\n\nStep 1: Data Loading -\n")
    train_df, test_df, rul_df = load_cmapss_data(dataset_name, data_path)

    print("\n\nStep 2: Data Preprocessing -")
    preprocessor = Preprocessor(max_rul=125, use_paper_sensors=True)    
    train_processed, train_degradation_onsets = preprocessor.fit_transform(train_df.copy())
    print(f"\n\nTraining Data Processed: {train_processed.shape}")    
    print("\n\nNow, for testing data...")
    test_with_rul = prepare_test_data(test_df.copy(), rul_df)
    test_processed = preprocessor.transform(test_with_rul.copy())
    print(f"\n\nTest Data Processed: {test_processed.shape}")

    print("\n\nStep 3: Sequence Creation and Data Augmentation -")     
    X_train, y_train, cond_train, train_engine_ids  = create_sequences(
        df=train_processed,
        sequence_length=sequence_length,
        selected_features=preprocessor.selected_features,
        degradation_onsets=None)
    X_test, y_test, cond_test, test_engine_ids = create_sequences(
        df=test_processed,
        sequence_length=sequence_length,
        selected_features=preprocessor.selected_features,
        degradation_onsets=None)
    y_test_seq = np.minimum(y_test, 125)
    split_idx = int(0.8 * len(X_train))
    X_val = X_train[split_idx:]
    y_val = y_train[split_idx:]
    cond_val = cond_train[split_idx:]    
    X_train = X_train[:split_idx]
    y_train = y_train[:split_idx]
    cond_train = cond_train[:split_idx]
    print(f"\n\nData Split:")
    print(f"  Training: {len(X_train)} Sequences")
    print(f"  Validation: {len(X_val)} Sequences")
    print(f"  Test: {len(X_test)} Sequences")   
    print(f"\n\nOriginal Training Samples: {len(X_train)}")
    X_train, y_train, cond_train = augment_sequences(
        X_train, y_train, cond_train, augmentation_factor=3)
    print(f"Augmented Training Samples: {len(X_train)}")

    print("\n\nStep 4: Model Building -\n")    
    model = build_comprehensive_rul_model(
        sequence_length=sequence_length,n_features=X_train.shape[2],
        use_mc_dropout=False, dropout_rate=0.48)
    print("\n\nModel Summary:")
    model.summary()

    print("\n\nStep 5: Model Training -\n")
    trainer = UncertaintyAwareTrainer(model, learning_rate=0.00015)
    try:
        from tensorflow.keras.optimizers.legacy import AdamW 
        trainer.optimizer = AdamW(
            learning_rate=0.00015, clipnorm=1.0)
    except:
        pass
    training_results = trainer.train(
        X_train, cond_train, y_train, X_val, cond_val, y_val,
        epochs=epochs, batch_size=64, verbose=1)
    
    print("\n\nStep 6: Training Evaluation -")
    predictions = predict_deterministic(model, X_test, cond_test)
    y_pred_mean = predictions['mean']
    # Temperature scaling for uncertainty
    y_pred_std = predictions['std'] * 6.0
    y_true_rul = np.loadtxt(f'{data_path}RUL_{dataset_name}.txt')
    last_preds = []
    last_stds = []
    for eng_id in np.unique(test_engine_ids):
        mask = (test_engine_ids == eng_id)
        last_preds.append(y_pred_mean[mask][-1])
        last_stds.append(y_pred_std[mask][-1])
    last_preds = np.array(last_preds)
    last_stds = np.array(last_stds)
    unique_ids = np.array(sorted(np.unique(test_engine_ids))).astype(int)
    y_true_aligned = y_true_rul[unique_ids - 1]
    if len(y_true_aligned) != len(last_preds):
        raise RuntimeError(f"Alignment mismatch: y_true={len(y_true_aligned)} vs preds={len(last_preds)}")
    test_rmse = np.sqrt(mean_squared_error(y_true_aligned, last_preds))
    lower_95 = last_preds - 1.96 * last_stds
    upper_95 = last_preds + 1.96 * last_stds
    coverage_95 = np.mean((y_true_aligned >= lower_95) & (y_true_aligned <= upper_95)) * 100
    crit_mask = y_true_aligned <= 30
    crit_rmse = (
        np.sqrt(mean_squared_error(y_true_aligned[crit_mask], last_preds[crit_mask]))
        if crit_mask.any() else np.nan)
    n_all = len(y_true_rul)
    n_pred = len(last_preds)
    n_skipped = n_all - n_pred
    print(f"\n\nRMSE Achieved: {test_rmse:.4f}")
    print(f"Learned Uncertainty (σ): {np.mean(last_stds):.2f} cycles")
    print(f"95% CI Coverage: {coverage_95:.1f}%")
    print(f"Critical Zone RMSE (RUL ≤ 30): {crit_rmse:.2f}")
    train_losses = training_results['train_losses']
    val_losses = training_results['val_losses']
    train_rmses = training_results['train_rmses']
    val_rmses = training_results['val_rmses']
    best_val_rmse = training_results['best_val_rmse']
    best_val_loss = training_results['best_val_loss']

    print("\n\nStep 7: Training History Visualization -\n")
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    axes[0].plot(train_losses, label='Train Loss (NLL)', linewidth=2, color='blue')
    axes[0].plot(val_losses, label='Val Loss (NLL)', linewidth=2, color='red')
    axes[0].axhline(y=best_val_loss, color='green', linestyle='--', 
                    label=f'Best Val Loss: {best_val_loss:.4f}')
    axes[0].set_xlabel('Epoch', fontsize=12)
    axes[0].set_ylabel('Loss (NLL)', fontsize=12)
    axes[0].set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)    
    axes[1].plot(train_rmses, label='Train RMSE', linewidth=2, color='blue')
    axes[1].plot(val_rmses, label='Val RMSE', linewidth=2, color='red')
    axes[1].axhline(y=best_val_rmse, color='green', linestyle='--', 
                    label=f'Best Val RMSE: {best_val_rmse:.2f}')
    axes[1].set_xlabel('Epoch', fontsize=12)
    axes[1].set_ylabel('RMSE', fontsize=12)
    axes[1].set_title('Training and Validation RMSE', fontsize=14, fontweight='bold')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{dataset_name}_training_history.png', dpi=300, bbox_inches='tight')
    plt.show()
    print(f"\n\nTraining history plot saved: {dataset_name}_training_history.png")
    print(f"\n\nTraining History Summary:")
    print(f"  Best Validation RMSE: {best_val_rmse:.2f}")
    print(f"  Best Validation NLL: {best_val_loss:.4f}")
    print(f"  Total Epochs: {len(train_losses)}")

    print("\n\nStep 8: Comprehensive Evaluation -")
    evaluator = ComprehensiveEvaluator()
    metrics = evaluator.evaluate_all(
        y_true=y_true_aligned, y_pred=last_preds,
        y_pred_std=last_stds, lower_bound=lower_95, upper_bound=upper_95)
    
    print("\n\nStep 9: False Positive, False Negative Analysis -")
    fp_fn_analyzer = MaintenanceDecisionAnalyzer(
        critical_threshold=30,
        warning_threshold=50)    
    fp_fn_metrics = fp_fn_analyzer.print_comprehensive_report(
        y_true=y_true_aligned, 
        y_pred=last_preds)    
    fp_fn_analyzer.plot_fp_fn_analysis(
        y_true=y_true_aligned,
        y_pred=last_preds,
        save_path=f'{dataset_name}_fp_fn_analysis.png')
        
    print("\n\nStep 10: Visualizations -")
    evaluator.plot_results(
        y_test, y_pred_mean, y_pred_std=y_pred_std,
        lower_bound=predictions['lower_95'], upper_bound=predictions['upper_95'],
        save_path=f'{dataset_name}_results.png') 
       
    print("\n\nStep 11: Final Report -")
    print(f"\n\nDataset: {dataset_name}")
    print(f"Sequence Length: {sequence_length}")    
    print(f"\n\nData Sizes:")
    print(f"  Training sequences: {len(X_train)}")
    print(f"  Validation sequences: {len(X_val)}")
    print(f"  Test sequences: {len(X_test)}")
    print(f"\n\nAccuracy Performance:")
    print(f"  RMSE: {metrics['RMSE']:.4f}")
    print(f"  MAE:  {metrics['MAE']:.4f}")
    print(f"  MAPE: {metrics['MAPE']:.2f}%")
    print(f"  R²:   {metrics['R2']:.4f}")
    print(f"\n\nTraining:")
    print(f"  Best Val RMSE: {best_val_rmse:.2f}")
    print(f"  Total Epochs: {len(train_losses)}")
    
    print("\n\nPipeline Completed Successfully.")
    predictions_df = pd.DataFrame({
        "true_rul": y_true_aligned, "Predicted_RUL": last_preds, "Uncertainty": last_stds,
        "lower_95": lower_95, "upper_95": upper_95})
    return model, metrics, predictions_df, fp_fn_metrics

In [ ]:
if __name__ == "__main__":

    # Configuration
    DATA_PATH = 'CMAPSS/'  # Update with your data path
    DATASET_NAME = 'FD001'  # Choose from: FD001, FD002, FD003, FD004
    SEQUENCE_LENGTH = 30
    EPOCHS = 250

    print("\nConfiguration:")
    print(f"  Dataset: {DATASET_NAME}")
    print(f"  Sequence Length: {SEQUENCE_LENGTH}")
    print(f"  Training Epochs: {EPOCHS}")
    print(f"  Data Path: {DATA_PATH if DATA_PATH else 'Current directory'}")  
    model, metrics, predictions, fp_fn_metrics = main_pipeline(
        dataset_name=DATASET_NAME, data_path=DATA_PATH,
        sequence_length=SEQUENCE_LENGTH, epochs=EPOCHS)
    
    print("\nMaintenance Recommendations:-")
    print("\nPredictions Summary:")
    print(predictions[['true_rul', 'Predicted_RUL']].head())
    predictions_sorted = predictions.sort_values('Predicted_RUL')
    print("\n- CRITICAL: Immediate Maintenance Required (RUL < 20 cycles)")
    critical = predictions_sorted[predictions_sorted['Predicted_RUL'] < 20]
    if len(critical) > 0:
        print(f"  Engines requiring immediate attention: {len(critical)}")
        print(f"  Average RUL: {critical['Predicted_RUL'].mean():.1f}")
    else:
        print("  No engines in critical state")
    print("\n- WARNING: Schedule Maintenance Soon (20 ≤ RUL < 50 cycles)")
    warning = predictions_sorted[
        (predictions_sorted['Predicted_RUL'] >= 20) & 
        (predictions_sorted['Predicted_RUL'] < 50)]
    if len(warning) > 0:
        print(f"  Engines requiring scheduled maintenance: {len(warning)}")
        print(f"  Average RUL: {warning['Predicted_RUL'].mean():.1f}")
    else:
        print("  No engines in warning state")
    print("\n- HEALTHY: Normal Operation (RUL ≥ 50 cycles)")
    healthy = predictions_sorted[predictions_sorted['Predicted_RUL'] >= 50]
    if len(healthy) > 0:
        print(f"  Engines in healthy state: {len(healthy)}")
        print(f"  Average RUL: {healthy['Predicted_RUL'].mean():.1f}")
    else:
        print("  No engines in healthy state")
    print("\n- High Uncertainty Alerts (uncertainty > 15% of predicted RUL)")
    high_uncertainty = predictions[
        (predictions['Uncertainty'] / (predictions['Predicted_RUL'] + 1e-10)) > 0.15]
    if len(high_uncertainty) > 0:
        print(f"  Engines with high prediction uncertainty: {len(high_uncertainty)}")
    else:
        print("  All predictions have acceptable uncertainty levels.")
    
    print("\nAnalysis Completed Successfully.")